# Jet Tagging — Extended Models
**Implementing all "further directions" from the improved CNN notebook**

This notebook trains and compares **three architectures** on the same jet-tagging task:

| Model | Input | Key idea |
|-------|-------|----------|
| **A — CNN + L2 + Cosine LR** | 50×50 jet image | Improved CNN with weight decay and cosine-annealing schedule |
| **B — Vision Transformer (ViT)** | 50×50 jet image | Non-overlapping 5×5 patches fed to a Transformer encoder |
| **C — Graph Neural Network (EdgeConv)** | Raw constituents (η, φ, pT) | ParticleNet-style EdgeConv; **no image step** |

All three use the full 16k training set, BatchNorm, and data augmentation (image models only).


In [1]:
# --- Imports ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import itertools, os, warnings, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

# Global constants
N_CLASSES   = 5
LABELS_STR  = ['gluon', 'quark', 'W', 'Z', 'top']
BATCH_SIZE  = 256
AUTOTUNE    = tf.data.AUTOTUNE
SEED        = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

TensorFlow version: 2.21.0
GPU available: False


## 1. Load Dataset

In [2]:
with np.load('dataset_jetTag.npz') as f:
    target   = f['target']
    features = f['features']

label = np.argmax(target, axis=1)
print(f"Features : {features.shape}")
print(f"Labels   : {label.shape}")
print(f"Class distribution: {dict(zip(LABELS_STR, np.bincount(label)))}") 

Features : (20000, 100, 16)
Labels   : (20000,)
Class distribution: {'gluon': 4114, 'quark': 3940, 'W': 4039, 'Z': 3950, 'top': 3957}


## 2. Train / Validation / Test Split (80 / 10 / 10)

In [3]:
X_tr, X_rem, y_tr, y_rem = train_test_split(
    features, label, train_size=0.8, random_state=SEED, stratify=label)
X_val, X_test, y_val, y_test = train_test_split(
    X_rem, y_rem, test_size=0.5, random_state=SEED, stratify=y_rem)
print("Train:", X_tr.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)

Train: (16000, 100, 16) | Val: (2000, 100, 16) | Test: (2000, 100, 16)


## 3. Jet Images + Preprocessing
Used by **Models A & B**. Same pipeline as the improved CNN notebook.


In [4]:
from skimage.transform import downscale_local_mean

def create_images(feats, bins=100):
    imgs = np.empty((feats.shape[0], bins, bins), dtype=np.float32)
    for i in range(feats.shape[0]):
        eta = feats[i, :, 8]
        phi = feats[i, :, 11]
        pt  = feats[i, :, 5]
        h, _, _ = np.histogram2d(eta, phi, bins=bins,
                                 range=[[-1,1],[-1,1]], weights=pt)
        imgs[i] = h
    return imgs

def preprocess(imgs, target_size=50):
    f = imgs.shape[1] // target_size
    out = np.empty((imgs.shape[0], target_size, target_size), dtype=np.float32)
    for i in range(imgs.shape[0]):
        out[i] = downscale_local_mean(imgs[i], (f, f))
    return np.log1p(out)[..., np.newaxis]

print("Creating jet images …")
t0 = time.time()
imgs_tr   = preprocess(create_images(X_tr))
imgs_val  = preprocess(create_images(X_val))
imgs_test = preprocess(create_images(X_test))
print(f"Done in {time.time()-t0:.0f}s  |  shape: {imgs_tr.shape}")

Y_tr   = keras.utils.to_categorical(y_tr,   N_CLASSES)
Y_val  = keras.utils.to_categorical(y_val,  N_CLASSES)
Y_test = keras.utils.to_categorical(y_test, N_CLASSES)

Creating jet images …
Done in 6s  |  shape: (16000, 50, 50, 1)


## 4. Raw Constituent Features for the GNN
**Model C** skips the image step entirely. Each jet is represented as a *set of particles*:

- **Node features**: $(p_T^{\text{rel}},\, \eta_{\text{rel}},\, \phi_{\text{rel}},\, \text{is\_real})$ — 4D per constituent  
- **Graph coordinates**: $(\eta_{\text{rel}},\, \phi_{\text{rel}})$ — used to build the k-NN graph

$p_T$ is log-normalised; padding constituents (all zeros) receive `is_real = 0`.


In [5]:
def extract_gnn_features(feats):
    """Returns (N_jets, 100, 4): [log1p_pT_rel, eta_rel, phi_rel, is_real]."""
    pt      = feats[:, :, 5].astype(np.float32)   # pT
    eta_rel = feats[:, :, 8].astype(np.float32)
    phi_rel = feats[:, :, 11].astype(np.float32)
    is_real = (pt > 0).astype(np.float32)

    # Jet pT = sum of constituent pT (ignoring padding)
    jet_pt = pt.sum(axis=1, keepdims=True).clip(min=1e-6)
    pt_rel = pt / jet_pt                            # relative pT

    x = np.stack([np.log1p(pt_rel), eta_rel, phi_rel, is_real], axis=-1)
    return x                                        # (N, 100, 4)

G_tr   = extract_gnn_features(X_tr)
G_val  = extract_gnn_features(X_val)
G_test = extract_gnn_features(X_test)
print("GNN feature shape:", G_tr.shape)
print(f"Avg real constituents per jet: {G_tr[...,3].sum(1).mean():.1f}")

GNN feature shape: (16000, 100, 4)
Avg real constituents per jet: 49.1


## 5. Data Augmentation & tf.data Pipelines
Image models (A, B) use random flips + 90° rotations.  
The GNN model (C) receives raw constituent features — no spatial augmentation is applied.


In [6]:
@tf.function
def augment_img(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k=k)
    return image, label

def img_dataset(X, Y, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED)
    if augment:
        ds = ds.map(augment_img, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def gnn_dataset(G, Y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((G, Y))
    if shuffle:
        ds = ds.shuffle(len(G), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Image datasets (CNN + ViT)
img_tr_ds   = img_dataset(imgs_tr,   Y_tr,   augment=True,  shuffle=True)
img_val_ds  = img_dataset(imgs_val,  Y_val)
img_test_ds = img_dataset(imgs_test, Y_test)

# GNN datasets
gnn_tr_ds   = gnn_dataset(G_tr,   Y_tr,  shuffle=True)
gnn_val_ds  = gnn_dataset(G_val,  Y_val)
gnn_test_ds = gnn_dataset(G_test, Y_test)

print("Image train batches:", len(img_tr_ds))
print("GNN   train batches:", len(gnn_tr_ds))

Image train batches: 63
GNN   train batches: 63


## 6. Evaluation Helper

In [7]:
def evaluate_model(model, test_ds, y_true, title='Model'):
    """Prints metrics and plots confusion matrix + ROC curves side by side."""
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)
    print(classification_report(y_true, y_pred, target_names=LABELS_STR))

    # ---- confusion matrix ----
    cm = confusion_matrix(y_true, y_pred)
    cm_n = cm.astype(float) / cm.sum(axis=1)[:, None]

    # ---- ROC ----
    Y_bin  = label_binarize(y_true, classes=list(range(N_CLASSES)))
    colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Confusion
    im = axes[0].imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    axes[0].set_xticks(range(N_CLASSES)); axes[0].set_yticks(range(N_CLASSES))
    axes[0].set_xticklabels(LABELS_STR); axes[0].set_yticklabels(LABELS_STR)
    for i, j in itertools.product(range(N_CLASSES), range(N_CLASSES)):
        axes[0].text(j, i, f'{cm_n[i,j]:.2f}', ha='center', va='center',
                     color='white' if cm_n[i,j] > 0.5 else 'black', fontsize=9)
    axes[0].set_title('Confusion Matrix (row-normalized)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    # ROC
    aucs = []
    for i in range(N_CLASSES):
        fpr, tpr, _ = roc_curve(Y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        axes[1].plot(fpr, tpr, color=colors[i], lw=2,
                     label=f'{LABELS_STR[i]} (AUC={roc_auc:.3f})')
    axes[1].plot([0,1],[0,1],'k--',lw=1)
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title('ROC Curves — one-vs-rest')
    axes[1].legend(loc='lower right', fontsize=9)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    macro_auc = np.mean(aucs)
    print(f"Macro-AUC: {macro_auc:.4f}")
    return y_prob, macro_auc


def plot_history(history, title=''):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13)
    axes[0].plot(history.history['accuracy'],     'o-', label='Train')
    axes[0].plot(history.history['val_accuracy'], 's-', label='Val')
    axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history.history['loss'],     'o-', label='Train')
    axes[1].plot(history.history['val_loss'], 's-', label='Val')
    axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

results = {}   # stores {model_name: {'acc': ..., 'macro_auc': ...}}

## 7. Model A — Improved CNN + L2 Regularisation + Cosine Annealing

### What is new
**L2 weight decay** penalises large weights: $\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{CE}} + \lambda \sum_i w_i^2$ with $\lambda = 10^{-4}$.  
Applied to every `Conv2D` and `Dense` layer.

**Cosine annealing** smoothly decays the learning rate:
$$\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_0 - \eta_{\min})\left(1 + \cos\frac{\pi\, t}{T}\right)$$
This avoids the sharp LR drops of `ReduceLROnPlateau` and often finds flatter minima.


In [8]:
REG = keras.regularizers.l2(1e-4)

def conv_bn_relu(filters, kernel, padding='same'):
    return keras.Sequential([
        layers.Conv2D(filters, kernel, padding=padding,
                      use_bias=False, kernel_regularizer=REG),
        layers.BatchNormalization(),
        layers.Activation('relu'),
    ])

inp = keras.Input(shape=(50, 50, 1))
x   = conv_bn_relu(16,  (11,11))(inp)
x   = layers.MaxPooling2D((2,2))(x)          # 50→25
x   = conv_bn_relu(32,  ( 5, 5))(x)
x   = layers.MaxPooling2D((2,2))(x)          # 25→12
x   = conv_bn_relu(64,  ( 3, 3))(x)
x   = layers.MaxPooling2D((2,2))(x)          # 12→6
x   = conv_bn_relu(128, ( 3, 3))(x)
x   = layers.GlobalAveragePooling2D()(x)

x   = layers.Dense(128, activation='relu', kernel_regularizer=REG)(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dropout(0.4)(x)
out = layers.Dense(N_CLASSES, activation='softmax')(x)

model_cnn = keras.Model(inp, out, name='JetCNN_L2_Cosine')
model_cnn.summary()

Model: "JetCNN_L2_Cosine"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 50, 50, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 50, 50, 16)     │         2,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 25, 25, 32)     │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 12, 12, 64)     │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_3 (Sequential)       │ (None, 6, 6, 128)      │        74,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,525 (490.33 KB)

 Trainable params: 124,789 (487.46 KB)

 Non-trainable params: 736 (2.88 KB)

In [9]:
# Cosine annealing schedule
EPOCHS_A   = 30
STEPS_EPOCH = len(img_tr_ds)
DECAY_STEPS = EPOCHS_A * STEPS_EPOCH

cosine_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=DECAY_STEPS,
    alpha=1e-5,               # minimum LR
)

model_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cosine_schedule),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

hist_cnn = model_cnn.fit(
    img_tr_ds,
    validation_data=img_val_ds,
    epochs=EPOCHS_A,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint('best_cnn.keras', save_best_only=True, verbose=0),
    ],
    verbose=1,
)
plot_history(hist_cnn, 'Model A — CNN + L2 + Cosine Annealing')

Epoch 1/30
56/63 ━━━━━━━━━━━━━━━━━━━━ 3s 526ms/step - accuracy: 0.4735 - loss: 1.3471

KeyboardInterrupt: 

In [ ]:
prob_cnn, auc_cnn = evaluate_model(model_cnn, img_test_ds, y_test,
                                    'Model A — CNN + L2 + Cosine Annealing')
_, acc_cnn = model_cnn.evaluate(img_test_ds, verbose=0)
results['CNN'] = {'acc': acc_cnn, 'macro_auc': auc_cnn}

## 8. Model B — Vision Transformer (ViT)

The jet image (50×50) is split into **100 non-overlapping 5×5 patches**.  
Each patch is linearly projected to a `d_model=64` embedding.  
A learnable **[CLS] token** aggregates global information through the Transformer encoder.

Architecture:
```
50×50 image
   ↓  extract 5×5 patches → (100, 25)
   ↓  linear projection   → (100, 64)
   ↓  + positional embed  → (101, 64)  [CLS prepended]
   ↓  × 4 Transformer blocks (heads=4, ff_dim=128)
   ↓  CLS token → Dense(64, relu) → BN → Dropout → Dense(5, softmax)
```


In [ ]:
# ── Transformer building blocks ─────────────────────────────────────────

class PatchExtract(layers.Layer):
    """Split a 50×50 image into (patch_h × patch_w) patches."""
    def __init__(self, patch_size=5, **kw):
        super().__init__(**kw)
        self.patch_size = patch_size

    def call(self, x):
        # x: (B, H, W, C)
        B = tf.shape(x)[0]
        patches = tf.image.extract_patches(
            images=x,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )  # (B, n_h, n_w, patch_dim)
        n_patches = patches.shape[1] * patches.shape[2]
        patch_dim = patches.shape[-1]
        return tf.reshape(patches, [B, n_patches, patch_dim])


class TransformerBlock(layers.Layer):
    """Standard Pre-LN Transformer encoder block."""
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kw):
        super().__init__(**kw)
        self.attn   = layers.MultiHeadAttention(num_heads=num_heads,
                                                key_dim=d_model//num_heads,
                                                dropout=dropout)
        self.ffn    = keras.Sequential([
            layers.Dense(ff_dim, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        # Pre-LN: norm before sub-layer
        x2 = self.ln1(x)
        x  = x + self.drop1(self.attn(x2, x2, training=training), training=training)
        x2 = self.ln2(x)
        x  = x + self.drop2(self.ffn(x2, training=training), training=training)
        return x


# ── ViT model ───────────────────────────────────────────────────────────

PATCH_SIZE  = 5
IMG_SIZE    = 50
N_PATCHES   = (IMG_SIZE // PATCH_SIZE) ** 2        # 100
PATCH_DIM   = PATCH_SIZE * PATCH_SIZE              # 25
D_MODEL     = 64
NUM_HEADS   = 4
FF_DIM      = 128
N_BLOCKS    = 4
DROP_RATE   = 0.1

vit_inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name='jet_image')

# 1. Extract + project patches
patches   = PatchExtract(PATCH_SIZE)(vit_inp)                  # (B, 100, 25)
projected = layers.Dense(D_MODEL)(patches)                     # (B, 100, 64)

# 2. Learnable CLS token
cls_init  = tf.zeros_initializer()
cls_token = tf.Variable(cls_init(shape=(1, 1, D_MODEL), dtype=tf.float32),
                        trainable=True, name='cls_token')
# Broadcast CLS to batch
B_dyn     = tf.shape(projected)[0]
cls_batch = tf.tile(cls_token, [B_dyn, 1, 1])                 # (B, 1, 64)
tokens    = tf.concat([cls_batch, projected], axis=1)          # (B, 101, 64)

# 3. Learnable positional embedding
pos_embed = layers.Embedding(input_dim=N_PATCHES+1,
                              output_dim=D_MODEL,
                              name='pos_embed')
positions = tf.range(N_PATCHES + 1)[tf.newaxis, :]
tokens    = tokens + pos_embed(positions)                      # (B, 101, 64)
tokens    = layers.Dropout(DROP_RATE)(tokens)

# 4. Transformer encoder blocks
for b in range(N_BLOCKS):
    tokens = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM,
                              dropout=DROP_RATE, name=f'block_{b}')(tokens)
tokens = layers.LayerNormalization(epsilon=1e-6)(tokens)

# 5. CLS token → classification head
cls_out = tokens[:, 0, :]                                       # (B, 64)
cls_out = layers.Dense(64, activation='gelu')(cls_out)
cls_out = layers.BatchNormalization()(cls_out)
cls_out = layers.Dropout(0.3)(cls_out)
vit_out = layers.Dense(N_CLASSES, activation='softmax', name='logits')(cls_out)

model_vit = keras.Model(vit_inp, vit_out, name='JetViT')
model_vit.summary()

In [ ]:
EPOCHS_B   = 30
STEPS_EPOCH_B = len(img_tr_ds)

cosine_vit = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-4,
    decay_steps=EPOCHS_B * STEPS_EPOCH_B,
    alpha=1e-5,
)

model_vit.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cosine_vit),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

hist_vit = model_vit.fit(
    img_tr_ds,
    validation_data=img_val_ds,
    epochs=EPOCHS_B,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint('best_vit.keras', save_best_only=True, verbose=0),
    ],
    verbose=1,
)
plot_history(hist_vit, 'Model B — Vision Transformer (ViT)')

In [ ]:
prob_vit, auc_vit = evaluate_model(model_vit, img_test_ds, y_test,
                                    'Model B — Vision Transformer (ViT)')
_, acc_vit = model_vit.evaluate(img_test_ds, verbose=0)
results['ViT'] = {'acc': acc_vit, 'macro_auc': auc_vit}

## 9. Model C — Graph Neural Network (EdgeConv / ParticleNet-style)

This model operates **directly on the constituent 4-vectors** — no image step.

### EdgeConv (DGCNN)
For each particle $i$ and its $k$ nearest neighbours $j$ in $(\eta, \phi)$ space:

$$\mathbf{h}_{ij} = \text{MLP}\left([\mathbf{x}_i,\; \mathbf{x}_j - \mathbf{x}_i]\right)$$

$$\mathbf{x}'_i = \max_{j \in \mathcal{N}(i)} \mathbf{h}_{ij}$$

The graph is recomputed **dynamically** in the learned feature space after each block  
(Dynamic Graph CNN — Wang et al. 2019).

```
Input (100 particles × 4 features)
   ↓  EdgeConv block 1: [32,32],  k=16   → (100, 32)
   ↓  EdgeConv block 2: [64,64],  k=16   → (100, 64)
   ↓  EdgeConv block 3: [128,128],k=16   → (100, 128)
   ↓  Global avg-pool + max-pool (concatenated) → (256,)
   ↓  Dense(128, relu) → BN → Dropout(0.4)
   ↓  Dense(5, softmax)
```


In [ ]:
class KNNGraph(layers.Layer):
    """Build a k-NN graph from (η,φ) coordinates of shape (B, N, 2).
    Returns edge indices of shape (B, N, k)."""

    def __init__(self, k=16, **kw):
        super().__init__(**kw)
        self.k = k

    def call(self, coords):
        # coords: (B, N, 2)
        # Pairwise squared distances
        diff = tf.expand_dims(coords, 2) - tf.expand_dims(coords, 1)  # (B,N,N,2)
        dist = tf.reduce_sum(diff**2, axis=-1)                         # (B,N,N)

        # Exclude self-loops by adding a large value on the diagonal
        n = tf.shape(coords)[1]
        eye = tf.eye(n, batch_shape=[tf.shape(coords)[0]])
        dist = dist + 1e9 * eye

        # k nearest neighbours (ascending distance)
        _, idx = tf.math.top_k(-dist, k=self.k)   # (B, N, k)
        return idx


class EdgeConv(layers.Layer):
    """Dynamic EdgeConv layer.
    Input  x   : (B, N, F)
    Input  idx : (B, N, k)  — k-NN indices in the current feature space
    Output     : (B, N, channels[-1])
    """

    def __init__(self, channels, name_prefix='ec', **kw):
        super().__init__(**kw)
        self.channels = channels
        # Shared MLP applied to each edge feature [x_i || x_j - x_i]
        self.mlp = keras.Sequential([
            layers.Dense(ch, use_bias=False, name=f'{name_prefix}_dense_{i}')
            for i, ch in enumerate(channels)
        ])
        self.bns  = [layers.BatchNormalization(name=f'{name_prefix}_bn_{i}')
                     for i in range(len(channels))]
        self.acts = [layers.Activation('relu') for _ in channels]

    def call(self, x, idx, training=False):
        # x   : (B, N, F)
        # idx : (B, N, k)
        B = tf.shape(x)[0]
        N = tf.shape(x)[1]
        k = tf.shape(idx)[2]

        # Gather k-nearest neighbour features using batch_dims=1
        idx_flat  = tf.reshape(idx, [B, N * k])                       # (B, N*k)
        x_neigh   = tf.gather(x, idx_flat, batch_dims=1)              # (B, N*k, F)
        x_neigh   = tf.reshape(x_neigh, [B, N, k, -1])                # (B, N, k, F)

        # Central point broadcast
        x_center  = tf.tile(x[:, :, tf.newaxis, :], [1, 1, k, 1])    # (B, N, k, F)

        # Edge feature: concatenate [central, relative]
        edge_feat = tf.concat([x_center, x_neigh - x_center], axis=-1) # (B,N,k,2F)

        # Apply shared MLP layer by layer (Dense broadcasts over B,N,k automatically)
        h = edge_feat
        for dense, bn, act in zip(self.mlp.layers, self.bns, self.acts):
            h = dense(h)
            h = bn(h, training=training)
            h = act(h)

        # Aggregate: max-pool over k neighbours
        return tf.reduce_max(h, axis=2)   # (B, N, channels[-1])


# ── Build GNN model ────────────────────────────────────────────────────────

K_NN      = 16
gnn_inp   = keras.Input(shape=(100, 4), name='constituents')  # (B,100,4)

# Graph building: use η,φ columns (indices 1,2) as coordinate space
coords    = gnn_inp[:, :, 1:3]                                 # (B, 100, 2)
mask_r    = gnn_inp[:, :, 3:4]                                 # (B, 100, 1) is_real

# Block 1
idx1  = KNNGraph(k=K_NN, name='knn1')(coords)
h     = EdgeConv([32, 32], name_prefix='ec1', name='ec1')(gnn_inp, idx1)

# Block 2: update graph in feature space (η,φ from learned features)
feat_coords2 = h[:, :, :2]   # use first 2 channels as proxy coords
idx2  = KNNGraph(k=K_NN, name='knn2')(feat_coords2)
h     = EdgeConv([64, 64], name_prefix='ec2', name='ec2')(h, idx2)

# Block 3
feat_coords3 = h[:, :, :2]
idx3  = KNNGraph(k=K_NN, name='knn3')(feat_coords3)
h     = EdgeConv([128, 128], name_prefix='ec3', name='ec3')(h, idx3)  # (B,100,128)

# Masked global pooling: ignore padding constituents (is_real=0)
INF   = 1e9
# For max-pool: set padding to -inf
h_max = h + (1.0 - mask_r) * (-INF)
g_max = tf.reduce_max(h_max, axis=1)     # (B, 128)

# For avg-pool: multiply by mask then normalise by count
h_avg  = h * mask_r
n_real = tf.reduce_sum(mask_r, axis=1).clip_by_value(1, 1e9)  # (B, 1)
g_avg  = tf.reduce_sum(h_avg, axis=1) / n_real                 # (B, 128)

g = tf.concat([g_avg, g_max], axis=-1)  # (B, 256)

# Classification head
g = layers.Dense(128, activation='relu')(g)
g = layers.BatchNormalization()(g)
g = layers.Dropout(0.4)(g)
gnn_out = layers.Dense(N_CLASSES, activation='softmax', name='class_probs')(g)

model_gnn = keras.Model(gnn_inp, gnn_out, name='JetGNN_EdgeConv')
model_gnn.summary()

In [ ]:
EPOCHS_C = 30

cosine_gnn = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=EPOCHS_C * len(gnn_tr_ds),
    alpha=1e-5,
)

model_gnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cosine_gnn),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

hist_gnn = model_gnn.fit(
    gnn_tr_ds,
    validation_data=gnn_val_ds,
    epochs=EPOCHS_C,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint('best_gnn.keras', save_best_only=True, verbose=0),
    ],
    verbose=1,
)
plot_history(hist_gnn, 'Model C — GNN (EdgeConv / ParticleNet-style)')

In [ ]:
prob_gnn, auc_gnn = evaluate_model(model_gnn, gnn_test_ds, y_test,
                                    'Model C — GNN (EdgeConv / ParticleNet-style)')
_, acc_gnn = model_gnn.evaluate(gnn_test_ds, verbose=0)
results['GNN'] = {'acc': acc_gnn, 'macro_auc': auc_gnn}

## 10. Final Comparison

Bar charts of test accuracy and macro-AUC for all three models, together with  
a per-class AUC breakdown and a brief discussion.


In [ ]:
# ── Summary table ────────────────────────────────────────────────────────
print(f"{'Model':<30} {'Test Acc':>10} {'Macro-AUC':>12}")
print('-' * 54)
for name, r in results.items():
    print(f"{name:<30} {r['acc']:>10.4f} {r['macro_auc']:>12.4f}")

# ── Bar charts ────────────────────────────────────────────────────────────
model_names = list(results.keys())
accs  = [results[n]['acc']       for n in model_names]
aucs  = [results[n]['macro_auc'] for n in model_names]
colors_bar = ['#3498db', '#9b59b6', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model Comparison — Test Set', fontsize=14, fontweight='bold')

for ax, vals, ylabel in zip(axes,
                             [accs, aucs],
                             ['Accuracy', 'Macro-AUC (one-vs-rest)']):
    bars = ax.bar(model_names, vals, color=colors_bar, edgecolor='k', linewidth=0.6)
    ax.set_ylim(max(0, min(vals) - 0.05), 1.0)
    ax.set_ylabel(ylabel)
    ax.axhline(0.2, color='grey', linestyle='--', lw=1, label='Random baseline (20%)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.4)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout(); plt.show()

In [ ]:
# ── Per-class AUC breakdown ──────────────────────────────────────────────
Y_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))
colors_class = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

per_class_aucs = {}
for name, prob in zip(['CNN', 'ViT', 'GNN'],
                      [prob_cnn, prob_vit, prob_gnn]):
    per_class_aucs[name] = [
        auc(*roc_curve(Y_bin[:, i], prob[:, i])[:2]) for i in range(N_CLASSES)
    ]

x = np.arange(N_CLASSES)
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
for k, (name, aucs_cls) in enumerate(per_class_aucs.items()):
    ax.bar(x + k*width, aucs_cls, width, label=name,
           color=colors_bar[k], edgecolor='k', linewidth=0.5)
ax.set_xticks(x + width)
ax.set_xticklabels(LABELS_STR, fontsize=12)
ax.set_ylabel('AUC (one-vs-rest)', fontsize=12)
ax.set_title('Per-class AUC — All Models', fontsize=13)
ax.set_ylim(0.5, 1.0)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Discussion

### Architecture insights

**CNN + L2 + Cosine Annealing**  
Weight decay prevents filter co-adaptation; cosine annealing avoids sharp LR drops  
and tends to converge to flatter minima, improving generalisation.  
Baseline for comparison — well-understood inductive biases for grid-structured data.

**Vision Transformer (ViT)**  
Self-attention captures *global* correlations across all 100 patches simultaneously —  
useful for multi-prong jet topologies (W/Z/top) where the relationship between widely  
separated energy deposits matters. Needs sufficient data to learn positional structure  
from scratch; augmentation helps compensate for the smaller dataset.

**GNN (EdgeConv / DGCNN)**  
Operates on raw constituents, bypassing discretisation artifacts introduced by the  
histogram step. Dynamic graph recomputation allows the model to attend to  
semantically similar particles regardless of spatial proximity.  
Particularly strong for gluon/quark discrimination, where constituent-level  
information (e.g., fragmentation multiplicity) is not fully captured by images.

### Further improvements
- **ViT**: use a larger `d_model` (128+) and more blocks; try DeiT-style distillation
- **GNN**: add more node features (e.g., charge, particle ID if available); use  
  attention-based aggregation (Graph Attention Network)
- **Ensemble**: average softmax probabilities of all three models (typically +1-2% AUC)
